# Where Q, K and V Come From — the Backward Pass

The companion to book chapter 07. Chapters 06, 07 and 09 built attention forwards;
this notebook differentiates it.

By the end you will have:
- Implemented the backward pass of scaled dot-product attention by hand, and checked
  it against `autograd` to floating-point agreement
- Verified the $\langle \partial L/\partial W^Q, W^Q\rangle =
  \langle \partial L/\partial W^K, W^K\rangle$ invariance that follows from
  $W^Q$ and $W^K$ only ever appearing as a product
- Measured how attention rows saturate as $d_k$ grows without the $\sqrt{d_k}$
  scaling, and found the damage shows up in the *distribution* of per-row gradients
  rather than in any aggregate norm
- Shown, by cutting the edge and watching the gradients vanish, that an
  encoder–decoder's encoder learns **only** through the cross-attention K and V
  projections

Everything here runs on CPU in well under a minute. No dataset is downloaded.

## Key Concept: there is no attention loss

`W^Q`, `W^K` and `W^V` are ordinary `nn.Linear` weights. They are initialized at
random, updated by the same optimizer step as every other parameter, and trained
jointly with the feed-forward head against **one scalar loss**. Nothing tells a query
what to look for. The interpretable heads in the papers are a by-product of minimizing
perplexity, not a thing anyone asked for.

So "where do Q, K and V come from?" is really a question about the backward pass: how
does a number computed at the output turn into a sensible change to a matrix twelve
layers away? The answer is six equations, and they are the subject of this notebook.

## Step 1: Imports

In [ ]:
import math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(0)
torch.set_printoptions(precision=6, sci_mode=False)

# Everything here is tiny; the CPU is the fastest option and the most reproducible.
device = torch.device('cpu')
print('torch', torch.__version__, '| device:', device)

## Step 2: the forward pass we are going to differentiate

One head, no batch axis, so the shapes stay readable:

| tensor | shape |
|---|---|
| `Q`, `K` | `(L, d_k)` |
| `V` | `(L, d_v)` |
| `S`, `P` | `(L, L)` |
| `O` | `(L, d_v)` |

In [ ]:
def attention_forward(Q, K, V, mask=None):
    """Returns the output plus the two intermediates the backward pass needs."""
    d_k = Q.shape[-1]
    S = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        S = S.masked_fill(mask, float('-inf'))
    P = torch.softmax(S, dim=-1)
    O = P @ V
    return O, P, S


L, d_k, d_v = 5, 8, 8
Q = torch.randn(L, d_k, requires_grad=True)
K = torch.randn(L, d_k, requires_grad=True)
V = torch.randn(L, d_v, requires_grad=True)

O, P, S = attention_forward(Q, K, V)
print('P rows sum to 1:', torch.allclose(P.sum(-1), torch.ones(L)))
print('O shape:', tuple(O.shape))

## Key Concept: the six equations

Given `G = dL/dO` arriving from above, the whole backward pass is:

$$\frac{\partial L}{\partial V} = P^{\top}G
\qquad
\frac{\partial L}{\partial P} = GV^{\top}$$

$$\frac{\partial L}{\partial S} = P \odot \big(\tfrac{\partial L}{\partial P}
   - \operatorname{rowsum}(\tfrac{\partial L}{\partial P} \odot P)\big)$$

$$\frac{\partial L}{\partial Q} = \frac{dS\,K}{\sqrt{d_k}}
\qquad
\frac{\partial L}{\partial K} = \frac{dS^{\top}Q}{\sqrt{d_k}}$$

The middle line is the row-wise softmax Jacobian $\operatorname{diag}(p) - pp^\top$
applied to the incoming gradient, written without ever building the $L \times L$
Jacobian itself. Every other line is a transpose-and-multiply.

## Step 3: implement the backward pass by hand

In [ ]:
def attention_backward(G, Q, K, V, P):
    """Hand-rolled backward for attention_forward. Returns (dQ, dK, dV)."""
    d_k = Q.shape[-1]
    # TODO:
    #   1. dV = P.T @ G                                    -> (L, d_v)
    #   2. dP = G @ V.T                                    -> (L, L)
    #   3. dS = P * (dP - rowsum(dP * P))                  -> (L, L)
    #      (this is diag(p) - p p^T applied to dP, one row at a time,
    #       without ever building the L x L Jacobian)
    #   4. dQ = dS @ K / sqrt(d_k)                         -> (L, d_k)
    #   5. dK = dS.T @ Q / sqrt(d_k)                       -> (L, d_k)
    pass

## Step 4: check it against autograd

In [ ]:
G = torch.randn(L, d_v)                    # a pretend gradient from the layer above

for t in (Q, K, V):
    t.grad = None
O, P, S = attention_forward(Q, K, V)
O.backward(G)                              # autograd's answer

dQ, dK, dV = attention_backward(G, Q.detach(), K.detach(), V.detach(), P.detach())

for name, ours, theirs in (('dQ', dQ, Q.grad), ('dK', dK, K.grad), ('dV', dV, V.grad)):
    err = (ours - theirs).abs().max().item()
    print(f'{name}: max abs difference = {err:.3e}   {"OK" if err < 1e-5 else "MISMATCH"}')

## Key Concept: every dS row sums to zero

$\sum_j p_j(g_j - p\cdot g) = p\cdot g - (p\cdot g)\sum_j p_j = 0$, because the
softmax weights sum to 1. This holds for **every** row of **every** softmax, always.

It is the single most useful check when you write your own attention backward: if a row
of `dS` does not sum to zero, the bug is in the softmax gradient and nowhere else.

In [ ]:
dP = G @ V.detach().transpose(-2, -1)
dS = P.detach() * (dP - (dP * P.detach()).sum(-1, keepdim=True))
print('row sums of dS:', dS.sum(-1))
print('max |row sum| =', dS.sum(-1).abs().max().item())

## Step 5: reproduce exercise 07.1 exactly

The book works this by hand with three tokens and $d_k = 2$. Same numbers here, so you
can check your pencil against the machine.

In [ ]:
Qe = torch.tensor([[1., 0.], [0., 1.], [1., 1.]], requires_grad=True)
Ke = Qe.detach().clone().requires_grad_(True)
Ve = Qe.detach().clone().requires_grad_(True)

Oe, Pe, _ = attention_forward(Qe, Ke, Ve)
Oe[0, 0].backward()                        # loss depends only on the first component of o1

print('P row 1      ', Pe[0].detach())     # book: [0.401, 0.198, 0.401]
print('o_1          ', Oe[0].detach())     # book: [0.802, 0.599]
print('dL/dq_1      ', Qe.grad[0])         # book: [0.1122, -0.0561]

g = torch.tensor([1., 0.])
dP_row = torch.stack([g @ Ve.detach()[j] for j in range(3)])
p = Pe.detach()[0]
dS_row = p * (dP_row - (p * dP_row).sum())
print('dP row 1     ', dP_row)             # book: [1, 0, 1]
print('p . g        ', (p * dP_row).sum().item())
print('dS row 1     ', dS_row)             # book: [0.0793, -0.1587, 0.0793]
print('dS row 1 sum ', dS_row.sum().item())

## Step 6: $W^Q$ and $W^K$ only ever appear as a product

The scores depend on the two matrices only through $W^Q(W^K)^\top$, so scaling one by
$c$ and the other by $1/c$ leaves the model **identical**. Differentiating that
invariance at $c = 1$ gives a relation the gradients must satisfy at every point in
training, on any data:

$$\langle \partial L/\partial W^Q,\ W^Q \rangle
= \langle \partial L/\partial W^K,\ W^K \rangle$$

Two things to take from the cell below. It is a real test of your own backward code.
And it means the norms of $W^Q$ and $W^K$ individually are not meaningful things to
monitor — the optimizer can slide along that flat direction without changing the
function at all.

In [ ]:
d_model = 16
X = torch.randn(6, d_model)
Wq = torch.randn(d_model, d_k, requires_grad=True)
Wk = torch.randn(d_model, d_k, requires_grad=True)
Wv = torch.randn(d_model, d_v, requires_grad=True)

def head_loss(Wq, Wk, Wv):
    O, _, _ = attention_forward(X @ Wq, X @ Wk, X @ Wv)
    return (O ** 2).sum()

loss = head_loss(Wq, Wk, Wv)
loss.backward()

lhs = (Wq.grad * Wq).sum().item()
rhs = (Wk.grad * Wk).sum().item()
print(f'<dL/dWq, Wq> = {lhs: .6f}')
print(f'<dL/dWk, Wk> = {rhs: .6f}')
print(f'difference   = {abs(lhs - rhs):.3e}')

# ... and the invariance itself: rescale the pair, get the same loss back.
with torch.no_grad():
    for c in (0.5, 2.0, 10.0):
        same = head_loss(Wq * c, Wk / c, Wv)
        print(f'c = {c:>4}: loss = {same.item():.6f}   (original {loss.item():.6f})')

## Step 7: the collapse the $\sqrt{d_k}$ scaling prevents

$\operatorname{Var}(q \cdot k) = d_k$, so raw scores spread as $\sqrt{d_k}$ and the
softmax rows sharpen as $d_k$ grows. As a row of `P` sharpens, `dS = p * (dP - p.dP)` dies
at both ends: the near-1 entry because the bracket goes to zero, every other entry because
`p` does. In the limit it is exact — a row that is *exactly* one-hot returns *exactly*
zero to every score in it.

Two measurements below. The first shows the scaling does control saturation. The second
shows where the damage actually lands, which is not where you might expect.

Note the initialization: `std = 1/sqrt(d_model)`, which is what makes `Q` and `K`
unit-variance. The $\operatorname{Var}(q\cdot k) = d_k$ derivation assumes that, and
with unit-variance *weights* instead the scores are enormous at every $d_k$ and there is
nothing left to measure.

In [ ]:
L_p, d_model_p, d_v_p = 16, 64, 16


def probe(d_k_value, scale, n_trials=32):
    """Mean row entropy of P, and the per-row norms of dL/dS, for random heads."""
    ent, row_norms, total, wq_norm = 0.0, [], 0.0, 0.0
    for t in range(n_trials):
        gen = torch.Generator().manual_seed(7000 + t)
        Xl = torch.randn(L_p, d_model_p, generator=gen)
        mk = lambda cols: (torch.randn(d_model_p, cols, generator=gen)
                           / math.sqrt(d_model_p)).requires_grad_()
        Wql, Wkl, Wvl = mk(d_k_value), mk(d_k_value), mk(d_v_p)
        Gl = torch.randn(L_p, d_v_p, generator=gen)      # fixed upstream gradient

        Sl = (Xl @ Wql) @ (Xl @ Wkl).T
        Sl = Sl / math.sqrt(d_k_value) if scale else Sl
        Sl.retain_grad()
        Pl = torch.softmax(Sl, dim=-1)
        (Pl @ (Xl @ Wvl)).backward(Gl)

        ent += (-(Pl * Pl.clamp_min(1e-12).log()).sum(-1)).mean().item()
        row_norms.append(Sl.grad.norm(dim=-1))
        total += Sl.grad.norm().item()
        wq_norm += Wql.grad.norm().item()
    return (ent / n_trials, torch.cat(row_norms),
            total / n_trials, wq_norm / n_trials)


dks = [2, 4, 8, 16, 32, 64, 128, 256]
scaled = [probe(d, True) for d in dks]
raw = [probe(d, False) for d in dks]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dks, [s[0] for s in scaled], 'o-', label=r'with $1/\sqrt{d_k}$')
ax.plot(dks, [r[0] for r in raw], 's--', label='without scaling')
ax.axhline(math.log(L_p), color='gray', ls=':', lw=1)
ax.text(2.2, math.log(L_p) - .12, 'uniform attention', fontsize=8, color='gray')
ax.set_xscale('log', base=2)
ax.set_xlabel('$d_k$'); ax.set_ylabel('mean row entropy of P (nats)')
ax.set_title('Does the attention row stay soft?')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f'{"d_k":>5} | {"entropy":>17} | {"||dS|| total":>17}')
print(f'{"":>5} | {"scaled":>8} {"raw":>8} | {"scaled":>8} {"raw":>8}')
print('-' * 47)
for d, s, r in zip(dks, scaled, raw):
    print(f'{d:>5} | {s[0]:>8.3f} {r[0]:>8.3f} | {s[2]:>8.3f} {r[2]:>8.3f}')

The entropy result is unambiguous: with the scaling, mean row entropy is flat in $d_k$;
without it, the rows collapse monotonically towards one-hot. That is the claim of chapter
06, measured.

The `||dS|| total` column is the interesting one, because it does **not** tell the same
story — the totals are within a small factor of each other, and the gradient reaching
`W^Q` is actually *larger* without the scaling. Aggregate norms are the wrong statistic
here. The next cell shows why.

In [ ]:
d_probe = 128
_, rows_scaled, tot_s, wq_s = probe(d_probe, True)
_, rows_raw, tot_r, wq_r = probe(d_probe, False)

def describe(name, rows, tot, wq):
    print(f'{name}')
    print(f'   per-row ||dS||   min {rows.min():.3e}   median {rows.median():.3e}'
          f'   max {rows.max():.3e}')
    print(f'   spread (max/min) {rows.max() / rows.min().clamp_min(1e-30):.3e}')
    print(f'   rows below 1e-4  {(rows < 1e-4).float().mean() * 100:.1f}%')
    print(f'   total ||dS|| {tot:.4f}      ||dL/dW^Q|| {wq:.3f}')

describe(f'with 1/sqrt(d_k)   (d_k={d_probe})', rows_scaled, tot_s, wq_s)
print()
describe(f'without scaling    (d_k={d_probe})', rows_raw, tot_r, wq_r)

fig, ax = plt.subplots(figsize=(7, 3.6))
bins = torch.logspace(-9, 1, 40)
ax.hist(rows_scaled.numpy(), bins=bins.numpy(), alpha=.65, label=r'with $1/\sqrt{d_k}$')
ax.hist(rows_raw.numpy(), bins=bins.numpy(), alpha=.65, label='without scaling')
ax.set_xscale('log')
ax.set_xlabel(r'$\|\partial L/\partial S\|$ for one attention row')
ax.set_ylabel('rows')
ax.set_title(f'Where the gradient goes, per row ($d_k$ = {d_probe})')
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Key Concept: the collapse is per-row, and averages hide it

Read the histogram, not the totals.

With the scaling, every row's gradient sits within a factor of about nine of every
other — the head is learning about all of its rows at once. Without it, that spread is
around **twelve orders of magnitude**: some rows still carry a healthy gradient, and
roughly 15% of them have saturated to one-hot and return numbers down at `1e-13`, which
is zero for any purpose the optimizer has.

That is why `||dS||` in aggregate looked fine. A Frobenius norm is dominated by its
largest entries, so a few loud surviving rows mask any number of dead ones, and
`||dL/dW^Q||` is *larger* without scaling because the surviving scores are enormous. A
model in this state is not failing to train — it is training on a shrinking subset of its
own attention pattern, which is a much harder thing to notice from a loss curve.

The practical version: when you suspect a head has collapsed, plot the **distribution** of
per-row attention entropy or per-row gradient norm. A mean will not show it, and neither
will a global gradient norm.

## Step 8: how the decoder's loss reaches the encoder

In an encoder–decoder the loss is computed entirely on the decoder's output. The encoder
has no objective of its own. Its **only** edges into the graph are the cross-attention K
and V projections:

$$\frac{\partial L}{\partial M} = \sum_{\ell}\left(
   \frac{\partial L}{\partial K_\ell}W_\ell^{K\top}
 + \frac{\partial L}{\partial V_\ell}W_\ell^{V\top}\right)$$

There is no $W^Q$ term — the query side of cross-attention is a decoder parameter. The
next two cells build a small translator and check that claim by cutting the edge.

In [ ]:
class TinyEncDec(nn.Module):
    """Two encoder layers, two decoder layers, deliberately minimal."""

    def __init__(self, vocab=20, d_model=32, n_heads=4, n_layers=2):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.enc = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model, n_heads, 64, batch_first=True)
            for _ in range(n_layers)
        ])
        self.dec = nn.ModuleList([
            nn.TransformerDecoderLayer(d_model, n_heads, 64, batch_first=True)
            for _ in range(n_layers)
        ])
        self.head = nn.Linear(d_model, vocab)

    def encode(self, src):
        h = self.emb(src)
        for layer in self.enc:
            h = layer(h)
        return h                                    # the memory M

    def decode(self, tgt, memory):
        T = tgt.shape[1]
        mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
        h = self.emb(tgt)
        for layer in self.dec:
            h = layer(h, memory, tgt_mask=mask)
        return self.head(h)


model = TinyEncDec()
src = torch.randint(0, 20, (4, 7))
tgt = torch.randint(0, 20, (4, 6))
criterion = nn.CrossEntropyLoss()


def grad_report(sever_memory: bool):
    """One forward/backward. Returns total grad norm for encoder and decoder params."""
    model.zero_grad(set_to_none=True)
    memory = model.encode(src)
    if sever_memory:
        memory = memory.detach()                    # cut the only edge to the encoder
    logits = model.decode(tgt[:, :-1], memory)
    loss = criterion(logits.reshape(-1, 20), tgt[:, 1:].reshape(-1))
    loss.backward()

    def total(prefix):
        s = 0.0
        for name, p in model.named_parameters():
            if name.startswith(prefix) and p.grad is not None:
                s += p.grad.norm().item() ** 2
        return s ** 0.5

    return loss.item(), total('enc'), total('dec')


loss, enc_n, dec_n = grad_report(sever_memory=False)
print(f'intact   : loss {loss:.4f}   encoder grad {enc_n:.6f}   decoder grad {dec_n:.6f}')

loss, enc_n, dec_n = grad_report(sever_memory=True)
print(f'severed  : loss {loss:.4f}   encoder grad {enc_n:.6f}   decoder grad {dec_n:.6f}')
print('\nSame loss, same decoder gradient, and the encoder receives exactly nothing.')

## Step 9: which projection carries it

`nn.TransformerDecoderLayer` names its cross-attention block `multihead_attn`, and packs
the three projections into one `in_proj_weight` of shape `(3*d_model, d_model)` — the
fused layout of the GPT-2 figure in chapter 06. Rows `0:d_model` are Q, `d_model:2*d_model`
are K, and the rest are V.

Split that gradient into thirds and the asymmetry of `cross_attn(tgt, memory, memory)`
becomes a number you can read.

In [ ]:
model.zero_grad(set_to_none=True)
memory = model.encode(src)
memory.retain_grad()                                # so we can look at dL/dM directly
logits = model.decode(tgt[:, :-1], memory)
criterion(logits.reshape(-1, 20), tgt[:, 1:].reshape(-1)).backward()

d = 32
print(f'||dL/dM|| = {memory.grad.norm().item():.6f}   (the encoder\'s entire signal)\n')
for i, layer in enumerate(model.dec):
    g = layer.multihead_attn.in_proj_weight.grad
    print(f'decoder layer {i} cross-attention:'
          f'  Q {g[:d].norm().item():.6f}'
          f'   K {g[d:2 * d].norm().item():.6f}'
          f'   V {g[2 * d:].norm().item():.6f}')

print('\nAll three are non-zero — but only K and V are computed *from the memory*,'
      '\nso only those two send anything back to the encoder. The Q gradient is'
      '\nformed from decoder hidden states and stays on the decoder side.')

## Exercises

Answers are folded up under each one. Write yours down before you open them — a check
you read first is not a check.

**1. Masking is free.** Add a causal mask to `attention_forward`, recompute `dS`, and
confirm the masked entries are *exactly* zero rather than merely small. Then explain why
`attention_backward` needed no change at all.

<details><summary>Answer</summary>
<p>A masked score is <code>-inf</code>, so <code>p_ij = 0</code> after the softmax, so
<code>dS_ij = p_ij * (dP_ij - p.g) = 0</code> — exactly, because it is a multiplication
by a hard zero and not a limit. Verified: <code>(dS[mask] == 0).all()</code> is
<code>True</code>, max absolute value <code>0.0</code>.</p>
<p>No backward code is needed because the same zero that removed the position from the
forward average removes it from the backward sum. The mask is enforced once, in the
forward pass, and the algebra carries it. This is also why masking <em>after</em> the
softmax is a real bug and not a stylistic choice: it would leave <code>p_ij != 0</code>,
and a future token would quietly send gradient to a past position's query.</p>
</details>

**2. The all-masked row.** Mask an entire row and watch `nan` appear. Which of the six
equations produces the first one — and are you sure it is one of the six?

<details><summary>Answer</summary>
<p>It is <strong>none of them</strong>. The <code>nan</code> is born in the
<em>forward</em> pass: every score in the row is <code>-inf</code>, so the softmax
denominator is <code>sum(exp(-inf)) = 0</code> and the row is <code>0/0 = nan</code>.
By the time any backward equation runs, <code>P</code> already contains
<code>nan</code>.</p>
<p>That matters for debugging. The instinct is to look at the backward pass because that
is where the damage shows up, but a printed <code>P</code> would have caught it one step
earlier. From there <code>nan</code> reaches every parameter on any path to the loss —
which, after one optimizer step, is effectively the whole model. Guard against
fully-padded sequences in the data pipeline; this bug is famous for surviving weeks of
training.</p>
</details>

**3. Multi-head.** Extend `attention_backward` to `(B, H, L, d_k)` inputs and check it
against autograd.

<details><summary>Answer</summary>
<p><strong>It already works — the function needs no edit.</strong> Every operation in it
is written with <code>transpose(-2, -1)</code> and <code>sum(-1)</code>, which address
the last two axes, and <code>@</code> broadcasts over any number of leading batch
dimensions. Checked at <code>B=2, H=3</code>: <code>dQ</code>, <code>dK</code> and
<code>dV</code> all match autograd to <code>1e-5</code>.</p>
<p>If you rewrote it and it broke, the near-certain cause is an absolute axis index —
<code>.T</code>, <code>transpose(0, 1)</code>, or <code>sum(1)</code> — which silently
means something different once batch and head axes are in front. Prefer negative indices
in any function that might one day be batched.</p>
</details>

**4. Post-norm versus pre-norm.** Build two 8-layer stacks differing only in
normalization placement, and compare the gradient reaching layer 1. Be careful how you
compare them.

<details><summary>Answer</summary>
<p>The trap is comparing raw gradient norms across the two architectures. Post-norm's
output is normalized and therefore bounded, while pre-norm's residual stream grows with
depth, so the two losses are on completely different scales — do it naively and pre-norm
appears to have a gradient about 500,000x larger, which is a fact about the loss and not
about the architecture.</p>
<p>The scale-free comparison is each block's gradient norm <em>relative to the top block
of the same stack</em>. Measured over 8 seeds, gradient at the bottom block as a fraction
of the top:</p>
<table>
<tr><th>depth</th><th>pre-norm</th><th>post-norm</th></tr>
<tr><td>8</td><td>0.74</td><td>0.16</td></tr>
<tr><td>16</td><td>0.59</td><td>0.10</td></tr>
</table>
<p>Both attenuate — pre-norm is not a free lunch — but post-norm attenuates about 4.5x
more at depth 8 and 5.6x more at depth 16, and its disadvantage grows with depth. That
matches the chapter's argument: post-norm puts a LayerNorm projection on the residual
path itself, once per block, so the identity term is not actually an identity.</p>
</details>

**5. Where a rare token's gradient goes.** After one backward pass, count the embedding
rows whose gradient is exactly zero. Relate the number to the batch.

<details><summary>Answer</summary>
<p>Exactly <code>V - (number of distinct token ids in the batch)</code>. With a vocabulary
of 50 and 23 distinct ids in the batch, 27 rows have exactly zero gradient — the embedding
is a lookup, so a row that was never looked up is on no path to the loss.</p>
<p>The consequence is that a rare word's vector is updated only on the handful of steps
where it appears, and sits at its initialization the rest of the time. That is why rare
embeddings stay close to random, why subword tokenizers exist, and why a bigger vocabulary
needs more <em>data</em> rather than merely more parameters — each new row needs its own
occurrences to learn from, and no amount of training on other tokens helps it.</p>
</details>